In [6]:
import pandas as pd
import sys
import os

sys.path.append(os.path.abspath('..'))

from SOLID.SRP_Detection_Final           import get_srp_report
from SOLID.OCP_Detection_Final           import get_ocp_report
from SOLID.Liskov_Substitution_Principle import get_lsp_report
from SOLID.ISP_detect                    import get_isp_report
from SOLID.dependancy_principle          import get_dip_report

In [7]:
DATA_PATH = r'python-codes-labeled.xlsx'

df = pd.read_excel(DATA_PATH, dtype=str)
df = df.dropna()
df.head(10)

,id,language,code,srp,ocp,lsp,isp,dip
0,1,Python,"class Writer:\n def __init__(self, type: in...",Violation,Violation,Pass,Pass,Violation
1,2,Python,class TelephoneDirectory:\n def __init__(self...,Pass,Pass,Pass,Pass,Violation
2,3,Python,import unittest\nclass Shape:\n def compute...,Pass,Pass,Pass,Pass,Pass
3,4,Python,"class StringEncrypter:\n def encrypt(self, ...",Pass,Pass,Pass,Pass,Violation
4,5,Python,import logging\nimport pandas as pd\nimport nu...,Pass,Pass,Pass,Pass,Pass
5,6,Python,"class Book:\n def __init__(self, title, aut...",Pass,Violation,Pass,Pass,Violation
6,7,Python,"from abc import ABC, abstractmethod\n\nclass U...",Violation,Pass,Violation,Violation,Pass


In [16]:
def extract_status(result):
    # list of dictionaries
    if isinstance(result, list):
        return [item.get('status') for item in result]

    # single dictionary
    elif isinstance(result, dict):
        return result.get('status')

    # fallback
    return None

In [17]:
df['SRP_detections'] = df['code'].apply(
    lambda x: extract_status(get_srp_report(x))
)

df['OCP_detections'] = df['code'].apply(
    lambda x: extract_status(get_ocp_report(x))
)

df['LSP_detections'] = df['code'].apply(
    lambda x: extract_status(get_lsp_report(x))
)

df['ISP_detections'] = df['code'].apply(
    lambda x: extract_status(get_isp_report(x))
)

df['DIP_detections'] = df['code'].apply(
    lambda x: extract_status(get_dip_report(x))
)
df.head(10)

,id,language,code,srp,ocp,lsp,isp,dip,SRP_detections,OCP_detections,LSP_detections,ISP_detections,DIP_detections
0,1,Python,"class Writer:\n def __init__(self, type: in...",Violation,Violation,Pass,Pass,Violation,[Pass],Violation,Pass,Violation,Pass
1,2,Python,class TelephoneDirectory:\n def __init__(self...,Pass,Pass,Pass,Pass,Violation,[Violation],Pass,Pass,Pass,Pass
2,3,Python,import unittest\nclass Shape:\n def compute...,Pass,Pass,Pass,Pass,Pass,"[Pass, Violation, Pass, Violation]",Pass,Pass,Pass,Violation
3,4,Python,"class StringEncrypter:\n def encrypt(self, ...",Pass,Pass,Pass,Pass,Violation,"[Pass, Pass, Pass]",Pass,Pass,Pass,Violation
4,5,Python,import logging\nimport pandas as pd\nimport nu...,Pass,Pass,Pass,Pass,Pass,"[Pass, Pass, Pass, Pass]",Pass,Pass,Pass,Violation
5,6,Python,"class Book:\n def __init__(self, title, aut...",Pass,Violation,Pass,Pass,Violation,"[Pass, Violation]",Pass,Pass,Pass,Pass
6,7,Python,"from abc import ABC, abstractmethod\n\nclass U...",Violation,Pass,Violation,Violation,Pass,"[Violation, Pass, Pass]",Pass,Pass,Violation,Pass


In [20]:
def normalize_label(value):
    # Handle lists like ['Pass']
    if isinstance(value, list):
        value = value[0]
        
    value = str(value).lower()

    if "violation" in value:
        return "Violation"

    return "Pass"

In [21]:
from sklearn.metrics import accuracy_score

srp_accuracy = accuracy_score(
    df['srp'].apply(normalize_label),
    df['SRP_detections'].apply(normalize_label)
)

ocp_accuracy = accuracy_score(
    df['ocp'].apply(normalize_label),
    df['OCP_detections'].apply(normalize_label)
)

lsp_accuracy = accuracy_score(
    df['lsp'].apply(normalize_label),
    df['LSP_detections'].apply(normalize_label)
)

isp_accuracy = accuracy_score(
    df['isp'].apply(normalize_label),
    df['ISP_detections'].apply(normalize_label)
)

dip_accuracy = accuracy_score(
    df['dip'].apply(normalize_label),
    df['DIP_detections'].apply(normalize_label)
)

print("SRP Accuracy:", srp_accuracy)
print("OCP Accuracy:", ocp_accuracy)
print("LSP Accuracy:", lsp_accuracy)
print("ISP Accuracy:", isp_accuracy)
print("DIP Accuracy:", dip_accuracy)

SRP Accuracy: 0.7142857142857143
OCP Accuracy: 0.8571428571428571
LSP Accuracy: 0.8571428571428571
ISP Accuracy: 0.8571428571428571
DIP Accuracy: 0.2857142857142857


In [23]:
all_actual = []
all_predicted = []

principles = ['SRP', 'OCP', 'LSP', 'ISP', 'DIP']

for p in principles:
    all_actual.extend(df[p.lower()].apply(normalize_label))
    all_predicted.extend(df[f'{p}_detections'].apply(normalize_label))

overall_accuracy = accuracy_score(all_actual, all_predicted)

print("Overall Accuracy:", overall_accuracy)

Overall Accuracy: 0.7142857142857143
